In [2]:
# 1. Upgrade pip just in case
!pip install --upgrade pip

# 2. Install a stable version of Rasterio that has pre-built binaries
# This bypasses the "GDAL/gdal-config" error by using a "wheel" file instead of compiling source code.
!pip install rasterio==1.3.10

# 3. Now install Sedona (It will see rasterio is already there and skip the bad step)
!pip install apache-sedona==1.6.1

# 4. Java is likely fine ("Nothing to do" means it's already installed), but we run it to be safe
!sudo yum install -y java-1.8.0-openjdk-devel
# --- IMPORTS & SETUP ---
import time
from pyspark.sql.functions import col, year, sum as _sum, desc, rank, to_timestamp, format_number
from pyspark.sql.window import Window
from project_setup import get_spark_session, CRIME_DATA, RE_CODES_DATA

spark = get_spark_session("Query2_Analysis")

Loaded plugins: dkms-build-requires, extras_suggestions, kernel-livepatch,
              : langpacks, priorities, update-motd, versionlock
amzn2-core                                               | 3.6 kB     00:00     
https://download.docker.com/linux/centos/2/x86_64/stable/repodata/repomd.xml: [Errno 14] HTTPS Error 404 - Not Found
Trying other mirror.
63 packages excluded due to repository priority protections
Package 1:java-1.8.0-openjdk-devel-1.8.0.472.b08-1.amzn2.0.1.x86_64 already installed and latest version
Nothing to do
Configuring Environment for 'Query2_Analysis'...
   Resource Config: 4 Executors | 1 Cores | 2g RAM
   JAVA_HOME set to: /usr/lib/jvm/java-1.8.0-openjdk-1.8.0.472.b08-1.amzn2.0.1.x86_64/jre
:: loading settings :: url = jar:file:/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.sedona#sedona-spark-shaded-3.4_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-50b7eefc-fbb0-4b79-85bf-558750f18c30;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1 in central
	found org.datasyslab#geotools-wrapper;1.6.1-28.2 in central
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 457ms :: artifacts dl 24ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1 from c

25/12/15 12:45:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
[Stage 0:>                                                          (0 + 1) / 1]

25/12/15 12:45:54 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/15 12:45:54 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/15 12:45:54 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/15 12:45:54 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/15 12:45:54 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
25/12/15 12:45:54 WARN SimpleFunctionRegistry: The function st_intersection_aggr replaced a previously registered function.
25/12/15 12:45:54 WARN SimpleFunctionRegistry: The function st_union_aggr replaced a previously registered function.
   Sedona Context Active


In [3]:
# ---WARMUP---
print("Warming up System (Loading data into OS Cache)...")
spark.read.option("header", "true").option("inferSchema", "true").csv(CRIME_DATA).count()
spark.read.option("header", "true").option("inferSchema", "true").csv(RE_CODES_DATA).count()
print("System is warm. Starting Fair Benchmark.\n")

Warming up System (Loading data into OS Cache)...
25/12/15 12:46:34 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


System is warm. Starting Fair Benchmark.



In [8]:
df_crime = spark.read.option("header", "true").option("inferSchema", "true").csv(CRIME_DATA)
df_crime.select("DATE OCC").show(3, truncate=False)

[Stage 28:==================================================>       (7 + 1) / 8]

+-----------------------+
|DATE OCC               |
+-----------------------+
|2010 Feb 20 12:00:00 AM|
|2010 Sep 12 12:00:00 AM|
|2010 Aug 09 12:00:00 AM|
+-----------------------+
only showing top 3 rows



In [4]:
# --- DATAFRAME API IMPLEMENTATION ---
print("Starting Query 2 (DataFrame API)...")
start_time = time.time()

df_crime = spark.read.option("header", "true").option("inferSchema", "true").csv(CRIME_DATA)
df_re_codes = spark.read.option("header", "true").option("inferSchema", "true").csv(RE_CODES_DATA)

df_processed = df_crime.withColumn("Year", year(to_timestamp(col("DATE OCC"), "yyyy MMM dd hh:mm:ss a")))
grouped_df = df_processed.filter(col("Year").isNotNull() & col("Vict Descent").isNotNull()) \
                         .groupBy("Year", "Vict Descent") \
                         .count().withColumnRenamed("count", "victims_count")

result_joined = grouped_df.join(df_re_codes, "Vict Descent", "inner")

window_year = Window.partitionBy("Year")
df_calc = result_joined.withColumn("total", _sum("victims_count").over(window_year)) \
                       .withColumn("percentage", (col("victims_count") / col("total")) * 100)

#Rank Top 3
window_rank = Window.partitionBy("Year").orderBy(desc("victims_count"))
df_final = df_calc.withColumn("rank", rank().over(window_rank)) \
                  .filter(col("rank") <= 3) \
                  .select(col("Year"), col("Vict Descent Full").alias("Desc"), col("victims_count"), format_number("percentage", 1).alias("%")) \
                  .orderBy(desc("Year"), desc("victims_count"))

print("\nDataFrame Results:")
df_final.show(20, truncate=False)
print(f"DF Execution Time: {time.time() - start_time:.2f} seconds")

Starting Query 2 (DataFrame API)...



DataFrame Results:


[Stage 18:==================================================>       (7 + 1) / 8]

+----+----------------------+-------------+----+
|Year|Desc                  |victims_count|%   |
+----+----------------------+-------------+----+
|2025|Hispanic/Latin/Mexican|34           |40.5|
|2025|Unknown               |24           |28.6|
|2025|White                 |13           |15.5|
|2024|Hispanic/Latin/Mexican|28576        |29.1|
|2024|White                 |22958        |23.3|
|2024|Unknown               |19984        |20.3|
|2023|Hispanic/Latin/Mexican|69401        |34.6|
|2023|White                 |44615        |22.2|
|2023|Black                 |30504        |15.2|
|2022|Hispanic/Latin/Mexican|73111        |35.6|
|2022|White                 |46695        |22.8|
|2022|Black                 |34634        |16.9|
|2021|Hispanic/Latin/Mexican|63676        |35.1|
|2021|White                 |44523        |24.5|
|2021|Black                 |30173        |16.6|
|2020|Hispanic/Latin/Mexican|61606        |35.3|
|2020|White                 |42638        |24.5|
|2020|Black         

In [5]:
# --- SQL API IMPLEMENTATION ---
print("Starting Query 2 (SQL API)...")
start_time = time.time()
df_crime = spark.read.option("header", "true").option("inferSchema", "true").csv(CRIME_DATA)
df_re_codes = spark.read.option("header", "true").option("inferSchema", "true").csv(RE_CODES_DATA)
df_crime.createOrReplaceTempView("crime_data")
df_re_codes.createOrReplaceTempView("re_codes")

result_sql = spark.sql("""
WITH ProcessedData AS (
    SELECT YEAR(TO_TIMESTAMP(`DATE OCC`, 'yyyy MMM dd hh:mm:ss a')) as Year, `Vict Descent`
    FROM crime_data WHERE `DATE OCC` IS NOT NULL
),
WithTotals AS (
    SELECT g.Year, r.`Vict Descent Full` as Description, COUNT(*) as cnt,
           SUM(COUNT(*)) OVER (PARTITION BY g.Year) as total_year
    FROM ProcessedData g
    JOIN re_codes r ON g.`Vict Descent` = r.`Vict Descent`
    GROUP BY g.Year, r.`Vict Descent Full`
)
SELECT Year, Description, cnt as `#`, ROUND((cnt/total_year)*100, 1) as `%`
FROM (
    SELECT *, RANK() OVER (PARTITION BY Year ORDER BY cnt DESC) as rnk FROM WithTotals
) WHERE rnk <= 3 ORDER BY Year DESC, cnt DESC
""")

print("\nSQL Results:")
result_sql.show(20, truncate=False)
print(f"SQL Execution Time: {time.time() - start_time:.2f} seconds")

Starting Query 2 (SQL API)...


25/12/15 12:48:20 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.

SQL Results:


[Stage 29:==================================================>       (7 + 1) / 8]

+----+----------------------+-----+----+
|Year|Description           |#    |%   |
+----+----------------------+-----+----+
|2025|Hispanic/Latin/Mexican|34   |40.5|
|2025|Unknown               |24   |28.6|
|2025|White                 |13   |15.5|
|2024|Hispanic/Latin/Mexican|28576|29.1|
|2024|White                 |22958|23.3|
|2024|Unknown               |19984|20.3|
|2023|Hispanic/Latin/Mexican|69401|34.6|
|2023|White                 |44615|22.2|
|2023|Black                 |30504|15.2|
|2022|Hispanic/Latin/Mexican|73111|35.6|
|2022|White                 |46695|22.8|
|2022|Black                 |34634|16.9|
|2021|Hispanic/Latin/Mexican|63676|35.1|
|2021|White                 |44523|24.5|
|2021|Black                 |30173|16.6|
|2020|Hispanic/Latin/Mexican|61606|35.3|
|2020|White                 |42638|24.5|
|2020|Black                 |28785|16.5|
|2019|Hispanic/Latin/Mexican|72458|36.4|
|2019|White                 |48863|24.5|
+----+----------------------+-----+----+
only showing top